In [0]:
%sql
CREATE or REPLACE TABLE  practies.gold_layer.dim_customer (
  
  CustomerID int,
  FirstName VARCHAR(20),
  LastName VARCHAR(20),
  Email VARCHAR(100),
  Domine_Name VARCHAR(20),
  PhoneNumber VARCHAR(20),
  JoinDate DATE,
  last_date TIMESTAMP)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

# 1. Define the function using the LOCAL session
def upsert_to_delta(microBatchDF, batchId):
    # IMPORTANT: Get the spark session from the micro-batch itself
    spark_local = microBatchDF.sparkSession
    
    # Check if the batch is empty to save processing time
    if microBatchDF.count() == 0:
        return

    # 2. Reference the table using the local session
    target_table = DeltaTable.forName(spark_local, "practies.gold_layer.dim_customer")
    
    # 3. Perform the Merge
    (target_table.alias("t")
        .merge(
            microBatchDF.alias("s"),
            "t.CustomerID = s.CustomerID"
        )
        .whenMatchedUpdate(set = {
            "FirstName": "s.FirstName",
            "LastName": "s.LastName",
            "Email": "s.Email",
            "Domine_Name": "s.Domine_Name",
            "PhoneNumber": "s.PhoneNumber",
            "JoinDate": "s.JoinDate",
            "last_date": current_timestamp()
        })
        .whenNotMatchedInsert(values = {
            "CustomerID": "s.CustomerID",
            "FirstName": "s.FirstName",
            "LastName": "s.LastName",
            "Email": "s.Email",
            "Domine_Name": "s.Domine_Name",
            "PhoneNumber": "s.PhoneNumber",
            "JoinDate": "s.JoinDate",
            "last_date": current_timestamp()
        })
        .execute()
    )

# 4. Run the Stream
(spark.readStream.option("ignoreChanges", "true")\
    .table("practies.silver_layer.silver_customer")
    .writeStream
    .foreachBatch(upsert_to_delta) # This calls our fixed function
    .option("checkpointLocation", "/Volumes/practies/gold_layer/container/day2")
    .trigger(availableNow=True)
    .start()
)


In [0]:
%sql
select * from  practies.gold_layer.dim_customer

In [0]:
# Replace this with the exact path you used in your 'checkpointLocation' option
checkpoint_path = "/Volumes/practies/gold_layer/container/day1"

# Delete the directory and all its contents
dbutils.fs.rm(checkpoint_path, recurse=True)

print(f"Checkpoint at {checkpoint_path} has been deleted.")
